## 08 — Build Gold: match_momentum (per-minute pressure × shot metrics)

Builds the central Gold table `match_momentum` — a per-minute data grid for each team in each match, linking the pressure index (Sportmonks) with shot and goal statistics (FotMob). 

Key design decisions:
- **Minute spine**: created as `sequence(1, match_end_minute)` — ensures complete coverage even for eventless minutes
- **NULL pressure**: kept as NULL (not imputed to 0) because missing pressure data ≠ zero activity
- **Own goals**: credited to the opponent's `goal_count`, excluded from the shooting team's `shot_count`/xG


### 1. Load source tables 

Reads 4 silver tables needed for building Gold: `silver.pressure` (pressure index from Sportmonks), `silver.fotmob_shots` (shots from FotMob), `silver.match_source_map` (match ID mapping between sources), and `silver.team_source_map` (team ID mapping).


In [ ]:
from pyspark.sql import functions as F

pressure_df = spark.table(
    "wsl_analytics.silver.pressure"
)

shots_df = spark.table(
    "wsl_analytics.silver.fotmob_shots"
)

match_map_df = spark.table(
    "wsl_analytics.silver.match_source_map"
)

team_map_df = spark.table(
    "wsl_analytics.silver.team_source_map"
)

### 2. Build home teams view 

From `match_source_map`, builds a home teams view: for each match a row with `sportmonks_team_id`, `fotmob_team_id`, and `location = "home"`. Needed to join FotMob shots (which have `fotmob_team_id`) with the pressure table (which uses `sportmonks_team_id`).


In [ ]:
home_teams_df = (
    match_map_df
    .select(
        "sportmonks_fixture_id",
        "fotmob_match_id",
        "match_date",

        F.col("sportmonks_home_team_id")
            .alias("sportmonks_team_id"),

        F.col("fotmob_home_team_id")
            .alias("fotmob_team_id"),

        F.col("sportmonks_home_team")
            .alias("sportmonks_team_name"),

        F.col("fotmob_home_team")
            .alias("fotmob_team_name"),

        F.lit("home")
            .alias("location")
    )
)

### 3. Build away teams view 

Same for away teams.


In [ ]:
away_teams_df = (
    match_map_df
    .select(
        "sportmonks_fixture_id",
        "fotmob_match_id",
        "match_date",

        F.col("sportmonks_away_team_id")
            .alias("sportmonks_team_id"),

        F.col("fotmob_away_team_id")
            .alias("fotmob_team_id"),

        F.col("sportmonks_away_team")
            .alias("sportmonks_team_name"),

        F.col("fotmob_away_team")
            .alias("fotmob_team_name"),

        F.lit("away")
            .alias("location")
    )
)

### 4. Union to match_teams 

Combines home and away views via `unionByName` — each match now has 2 rows, one per team. This DataFrame is the central key connecting both ID spaces.


In [ ]:
match_teams_df = (
    home_teams_df
    .unionByName(
        away_teams_df
    )
)

display(match_teams_df)

### 5. Build opponent map 

Self-join of `match_teams_df` on `sportmonks_fixture_id` (excluding self via `!=`) creates for each team its `opponent_sportmonks_team_id`. Required for correct own-goal attribution.


In [ ]:
opponent_map_df = (
    match_teams_df.alias("team")
    .join(
        match_teams_df.alias("opp"),
        (
            F.col("team.sportmonks_fixture_id")
            ==
            F.col("opp.sportmonks_fixture_id")
        )
        &
        (
            F.col("team.sportmonks_team_id")
            !=
            F.col("opp.sportmonks_team_id")
        ),
        how="inner"
    )
    .select(
        F.col("team.sportmonks_fixture_id")
            .alias("sportmonks_fixture_id"),

        F.col("team.sportmonks_team_id")
            .alias("sportmonks_team_id"),

        F.col("opp.sportmonks_team_id")
            .alias("opponent_sportmonks_team_id")
    )
)


### 6. Map shots to Sportmonks IDs 

Joins `silver.fotmob_shots` with `match_teams_df` to translate `fotmob_match_id`/`fotmob_team_id` to `sportmonks_fixture_id`/`sportmonks_team_id`. **is_goal fix**: the `is_goal` field in Silver was NULL for all rows — here it is re-derived from `event_type == "goal"` (with lowercase and trim to normalise any whitespace).


In [ ]:
shots_mapped_df = (
    shots_df.alias("s")

    .join(
        match_teams_df.alias("m"),
        (
            F.col("s.fotmob_match_id")
            ==
            F.col("m.fotmob_match_id")
        )
        &
        (
            F.col("s.fotmob_team_id")
            ==
            F.col("m.fotmob_team_id")
        ),
        how="inner"
    )

    .select(
        F.col("m.sportmonks_fixture_id"),
        F.col("m.sportmonks_team_id"),
        F.col("m.fotmob_match_id"),
        F.col("m.fotmob_team_id"),

        F.col("s.shot_id"),
        F.col("s.fotmob_player_id"),
        F.col("s.player_name"),

        F.col("s.minute"),
        F.col("s.minute_added"),

        F.col("s.event_type"),
        F.col("s.xg"),
        F.col("s.xgot"),

        F.col("s.is_on_target"),
        F.col("s.is_blocked"),
        F.col("s.is_own_goal"),

        F.col("s.shot_type"),
        F.col("s.situation"),
        F.col("s.period")
    )

    # Defensive derivation: in the current Silver table `is_goal`
    # was NULL for all rows even though event_type == "Goal".
    .withColumn(
        "is_goal",
        F.lower(
            F.trim(F.col("event_type"))
        ) == F.lit("goal")
    )
)


### 7. Preview mapped shots 

Displays shots with Sportmonks IDs.


In [ ]:
display(shots_mapped_df)

### 8. Compute match_minute 

`match_minute = minute + coalesce(minute_added, 0)` — the event minute including added time. `coalesce` ensures missing added time is treated as 0 rather than NULL.


In [ ]:
shots_mapped_df = (
    shots_mapped_df
    .withColumn(
        "match_minute",
        F.col("minute")
        +
        F.coalesce(
            F.col("minute_added"),
            F.lit(0)
        )
    )
)


### 9. Add opponent + own-goal attribution 

Joins with `opponent_map_df` and creates `goal_credited_team_id`:
- regular goal: credited to the shooting team (`sportmonks_team_id`)
- own goal: credited to the opponent team (`opponent_sportmonks_team_id`)
- no goal: NULL


In [ ]:
shots_with_opponent_df = (
    shots_mapped_df
    .join(
        opponent_map_df,
        on=[
            "sportmonks_fixture_id",
            "sportmonks_team_id"
        ],
        how="left"
    )

    # A normal goal is credited to the shot-event team.
    # An own goal is credited to that team's opponent.
    .withColumn(
        "goal_credited_team_id",
        F.when(
            F.col("is_goal")
            &
            F.coalesce(
                F.col("is_own_goal"),
                F.lit(False)
            ),
            F.col("opponent_sportmonks_team_id")
        )
        .when(
            F.col("is_goal"),
            F.col("sportmonks_team_id")
        )
        .otherwise(
            F.lit(None).cast("long")
        )
    )
)


### 10. Exclude own-goal events from shot metrics 

Filters out `is_own_goal == True` before computing shot_count/xG — own goals are not the result of the team's offensive activity and should not count toward its attack metrics.


In [ ]:
# Own-goal events should not be counted as attacking shots/xG
# for the team whose player scored the own goal.
regular_shots_df = (
    shots_with_opponent_df
    .filter(
        ~F.coalesce(
            F.col("is_own_goal"),
            F.lit(False)
        )
    )
)


### 11. Preview minute display 

Preview of shots ordered by fixture and minute — verifies `match_minute` correctness.


In [ ]:
display(
    shots_mapped_df
    .select(
        "sportmonks_fixture_id",
        "player_name",
        "minute",
        "minute_added",
        "match_minute",
        "xg"
    )
    .orderBy(
        "sportmonks_fixture_id",
        "match_minute"
    )
)

### 12. Aggregate shots per minute 

Groups by (`sportmonks_fixture_id`, `sportmonks_team_id`, `match_minute`) and computes: shot count, xG sum, xGoT sum, max xG in the minute, and on-target shot count. `countDistinct(shot_id)` is safer than `count(*)` in case of source duplicates.


In [ ]:
shot_metrics_minute_df = (
    regular_shots_df
    .groupBy(
        "sportmonks_fixture_id",
        "sportmonks_team_id",
        "match_minute"
    )
    .agg(
        F.countDistinct(
            "shot_id"
        ).alias("shot_count"),

        F.sum(
            F.coalesce(
                F.col("xg"),
                F.lit(0.0)
            )
        ).alias("shot_xg"),

        F.sum(
            F.coalesce(
                F.col("xgot"),
                F.lit(0.0)
            )
        ).alias("shot_xgot"),

        F.max(
            "xg"
        ).alias("max_shot_xg"),

        F.sum(
            F.when(
                F.col("is_on_target") == True,
                1
            ).otherwise(0)
        ).alias("shots_on_target")
    )
)


### 13. Aggregate goals per minute 

Counts goals **after redistribution** — uses `goal_credited_team_id` (not `sportmonks_team_id`) so own goals are counted for the opponent.


In [ ]:
goals_minute_df = (
    shots_with_opponent_df
    .filter(
        F.col("is_goal") == True
    )
    .groupBy(
        "sportmonks_fixture_id",
        F.col("goal_credited_team_id")
            .alias("sportmonks_team_id"),
        "match_minute"
    )
    .agg(
        F.countDistinct(
            "shot_id"
        ).alias("goal_count")
    )
)


### 14. Aggregate own-goal events 

Separately counts own-goal events for the team whose player scored the own goal — for analytical and debugging purposes.


In [ ]:
own_goal_events_minute_df = (
    shots_with_opponent_df
    .filter(
        F.coalesce(
            F.col("is_own_goal"),
            F.lit(False)
        )
    )
    .groupBy(
        "sportmonks_fixture_id",
        "sportmonks_team_id",
        "match_minute"
    )
    .agg(
        F.countDistinct(
            "shot_id"
        ).alias("own_goal_event_count")
    )
)


### 15. Join shots + goals + own_goals 

Full outer join of the three minute-level DataFrames ensures no events are lost. `coalesce` on all metrics converts NULL to 0 — absence of an event in a minute is a true zero, not missing data.


In [ ]:
shots_minute_df = (
    shot_metrics_minute_df
    .join(
        goals_minute_df,
        on=[
            "sportmonks_fixture_id",
            "sportmonks_team_id",
            "match_minute"
        ],
        how="full"
    )
    .join(
        own_goal_events_minute_df,
        on=[
            "sportmonks_fixture_id",
            "sportmonks_team_id",
            "match_minute"
        ],
        how="full"
    )
    .withColumn(
        "shot_count",
        F.coalesce(F.col("shot_count"), F.lit(0))
    )
    .withColumn(
        "shot_xg",
        F.coalesce(F.col("shot_xg"), F.lit(0.0))
    )
    .withColumn(
        "shot_xgot",
        F.coalesce(F.col("shot_xgot"), F.lit(0.0))
    )
    .withColumn(
        "shots_on_target",
        F.coalesce(F.col("shots_on_target"), F.lit(0))
    )
    .withColumn(
        "goal_count",
        F.coalesce(F.col("goal_count"), F.lit(0))
    )
    .withColumn(
        "own_goal_event_count",
        F.coalesce(F.col("own_goal_event_count"), F.lit(0))
    )
)


### 16. Prepare pressure per minute 

Selects and casts columns from `silver.pressure`. If the API returned more than one pressure record for the same minute and team (which can occur), we take the maximum.


In [ ]:
pressure_minute_df = (
    pressure_df
    .select(
        F.col("fixture_id")
            .alias("sportmonks_fixture_id"),

        F.col("team_id")
            .alias("sportmonks_team_id"),

        F.col("minute")
            .cast("int")
            .alias("minute"),

        F.col("pressure")
            .cast("double")
            .alias("pressure")
    )
)

### 17. Aggregate duplicate pressure per minute 

MAX(pressure)` at (`fixture`, `team`, `minute`) level eliminates pressure duplicates without data loss.


In [ ]:
pressure_minute_df = (
    pressure_minute_df

    .groupBy(
        "sportmonks_fixture_id",
        "sportmonks_team_id",
        "minute"
    )

    .agg(
        F.max("pressure")
        .alias("pressure")
    )
)

### 18. Determine match end minute 

For each match determines `match_end_minute` as the maximum of: `90` (minimum match length), maximum pressure minute, and maximum shot minute. This ensures the minute spine covers injury time and potential extra time.


In [ ]:
pressure_max_minute_df = (
    pressure_minute_df
    .groupBy(
        "sportmonks_fixture_id"
    )
    .agg(
        F.max("minute")
        .alias("pressure_max_minute")
    )
)

In [ ]:
shots_max_minute_df = (
    shots_mapped_df
    .groupBy(
        "sportmonks_fixture_id"
    )
    .agg(
        F.max("match_minute")
        .alias("shots_max_minute")
    )
)

In [ ]:
match_end_df = (
    match_map_df
    .select(
        "sportmonks_fixture_id"
    )

    .join(
        pressure_max_minute_df,
        on="sportmonks_fixture_id",
        how="left"
    )

    .join(
        shots_max_minute_df,
        on="sportmonks_fixture_id",
        how="left"
    )

    .withColumn(
        "match_end_minute",

        F.greatest(
            F.lit(90),

            F.coalesce(
                F.col("pressure_max_minute"),
                F.lit(90)
            ),

            F.coalesce(
                F.col("shots_max_minute"),
                F.lit(90)
            )
        )
    )
)

### 19. Preview match end minutes 

Displays `match_end_minute` per match — verifies extra time and injury time are correctly detected.


In [ ]:
display(match_end_df)

### 20. Build minute spine 

`F.explode(F.sequence(1, match_end_minute))` creates one row per minute per team per match — a complete temporal grid. This is the base for left-joining pressure and shots, so event-free minutes also have rows (with NULL pressure and 0 shots).


In [ ]:
minute_spine_df = (
    match_teams_df

    .join(
        match_end_df,
        on="sportmonks_fixture_id",
        how="inner"
    )

    .withColumn(
        "minute",

        F.explode(
            F.sequence(
                F.lit(1),
                F.col("match_end_minute")
            )
        )
    )
)

### 21. Preview spine 

Displays the minute spine in chronological order.


In [ ]:
display(
    minute_spine_df
    .orderBy(
        "sportmonks_fixture_id",
        "sportmonks_team_id",
        "minute"
    )
)

### 22. Join spine + pressure 

Left joins the minute spine with pressure. Minutes without pressure remain in the table with `pressure = NULL` (intentional — not imputed to 0).


In [ ]:
momentum_df = (
    minute_spine_df.alias("base")

    .join(
        pressure_minute_df.alias("p"),

        (
            F.col("base.sportmonks_fixture_id")
            ==
            F.col("p.sportmonks_fixture_id")
        )
        &
        (
            F.col("base.sportmonks_team_id")
            ==
            F.col("p.sportmonks_team_id")
        )
        &
        (
            F.col("base.minute")
            ==
            F.col("p.minute")
        ),

        how="left"
    )

    .select(
        F.col("base.*"),
        F.col("p.pressure")
    )
)

### 23. Join + shots/goals 

Left joins with the per-minute shot/goal aggregation. Minutes without shots will have NULL in shot columns (converted to 0 in the next step).


In [ ]:
momentum_df = (
    momentum_df.alias("base")

    .join(
        shots_minute_df.alias("s"),

        (
            F.col("base.sportmonks_fixture_id")
            ==
            F.col("s.sportmonks_fixture_id")
        )
        &
        (
            F.col("base.sportmonks_team_id")
            ==
            F.col("s.sportmonks_team_id")
        )
        &
        (
            F.col("base.minute")
            ==
            F.col("s.match_minute")
        ),

        how="left"
    )

    .select(
        F.col("base.*"),

        F.col("s.shot_count"),
        F.col("s.shot_xg"),
        F.col("s.shot_xgot"),
        F.col("s.max_shot_xg"),
        F.col("s.shots_on_target"),
        F.col("s.goal_count"),
        F.col("s.own_goal_event_count")
    )
)

### 24. Impute zeros + derive flags 

Imputes 0 for NULL in all shot/goal columns (no event = true zero). Creates binary flags `has_shot`, `has_shot_on_target`, `has_goal`, and `pressure_available` (whether pressure is non-NULL).


In [ ]:
momentum_df = (
    momentum_df

    # Absence of a shot/goal event in a minute is a real zero.
    # Pressure is intentionally NOT imputed: NULL pressure != zero pressure.
    .withColumn(
        "shot_count",
        F.coalesce(F.col("shot_count"), F.lit(0))
    )
    .withColumn(
        "shot_xg",
        F.coalesce(F.col("shot_xg"), F.lit(0.0))
    )
    .withColumn(
        "shot_xgot",
        F.coalesce(F.col("shot_xgot"), F.lit(0.0))
    )
    .withColumn(
        "shots_on_target",
        F.coalesce(F.col("shots_on_target"), F.lit(0))
    )
    .withColumn(
        "goal_count",
        F.coalesce(F.col("goal_count"), F.lit(0))
    )
    .withColumn(
        "own_goal_event_count",
        F.coalesce(F.col("own_goal_event_count"), F.lit(0))
    )

    .withColumn(
        "has_shot",
        F.col("shot_count") > 0
    )
    .withColumn(
        "has_shot_on_target",
        F.col("shots_on_target") > 0
    )
    .withColumn(
        "has_goal",
        F.col("goal_count") > 0
    )
    .withColumn(
        "pressure_available",
        F.col("pressure").isNotNull()
    )
)


### 25. Data quality pre-write checks 

Before overwriting the Gold table, runs two checks:
1. **Goal audit**: total goal count and own-goal events — to compare against match scorelines
2. **Null audit**: counts NULLs in columns that should always be 0/1 (shots, goals, flags)


In [ ]:
# Data-quality checks before overwriting Gold
goal_audit_df = momentum_df.agg(
    F.sum("goal_count").alias("gold_goal_count"),
    F.sum("own_goal_event_count").alias("own_goal_events"),
    F.sum(
        F.when(F.col("has_goal"), 1).otherwise(0)
    ).alias("team_minutes_with_goal")
)

display(goal_audit_df)

null_audit_df = momentum_df.select(
    *[
        F.sum(
            F.when(F.col(c).isNull(), 1).otherwise(0)
        ).alias(c)
        for c in [
            "shot_count",
            "shot_xg",
            "shot_xgot",
            "shots_on_target",
            "goal_count",
            "own_goal_event_count",
            "has_shot",
            "has_shot_on_target",
            "has_goal"
        ]
    ]
)

display(null_audit_df)


### 26. Preview shot minutes 

Displays only minutes where a shot was actually taken — quick data quality check.


In [ ]:
display(
    momentum_df

    .filter(
        F.col("has_shot")
    )

    .select(
        "match_date",
        "sportmonks_fixture_id",
        "sportmonks_team_name",
        "minute",
        "pressure",
        "shot_count",
        "shot_xg",
        "max_shot_xg",
        "shot_xgot",
        "has_goal"
    )

    .orderBy(
        "sportmonks_fixture_id",
        "minute"
    )
)

### 27. Write gold.match_momentum 

Writes the Gold table. `mode("overwrite")` replaces the previous version — with every data fix (like own-goal correction) the entire table is rebuilt from scratch.


In [ ]:
GOLD_TABLE = (
    "wsl_analytics.gold.match_momentum"
)

(
    momentum_df
    .write
    .mode("overwrite")
    .saveAsTable(
        GOLD_TABLE
    )
)

### 28. SQL verify 

Reads the table via SQL ordered by fixture, team, and minute — final structure verification.


In [ ]:
%sql

SELECT *
FROM wsl_analytics.gold.match_momentum
ORDER BY
    sportmonks_fixture_id,
    sportmonks_team_id,
    minute;